Author: Emma Martinez

Course: ST 554 Project 2

Purpose: Project 2- Spark Data Quality & NFL Analysis

# Project #2

This notebook has two parts:

- **Part I** – Demonstrates the custom `SparkDataCheck` class built in `SparkDataCheck.py`. We load air-quality sensor data, run validation checks, and produce numeric/categorical summaries.
- **Part II** – Analyses NFL quarterback statistics (2005-2023) using both the pandas-on-Spark API and the Spark SQL DataFrame API side-by-side so we can compare the two styles.

We start every section by creating (or reusing) a `SparkSession`.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataQualityProject") \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 09:31:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 09:31:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


---
## Part I – The `SparkDataCheck` Class

### 1.1 Import the module

We import the custom class. The `importlib.reload` function is useful for when we edit `SparkDataCheck.py` and want the notebook to pick up the changes without restarting the kernel.

In [2]:
import importlib
import SparkDataCheck as sdc_module
importlib.reload(sdc_module)

from SparkDataCheck import SparkDataCheck

### 1.2 Load air-quality data using `from_csv`

The dataset contains hourly sensor readings from an Italian city. Columns include pollutant concentrations (CO, NMHC, C6H6, NOx, NO2), sensor responses, temperature (`T`), relative humidity (`RH`), and absolute humidity (`AH`).

In [3]:
AIR_PATH = 'air.csv'

air_obj = SparkDataCheck.from_csv(spark, AIR_PATH)
air_obj.df.printSchema()
air_obj.df.show(5)

root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-

26/03/26 09:31:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-emarti33@ncsu.edu/air.csv


### 1.3 Validation method demos

Each validation method modifies the `df` attribute of the object and returns `self`, allowing calls to be chained. Results are viewed by calling `.show()` on the `df` attribute directly.

#### `check_numeric_range` – five examples

In [4]:
# Example 1 – temperature between -5 and 45 degrees (both bounds provided)
air_obj.check_numeric_range('T', lower=-5, upper=45)
air_obj.df.select('T', 'T_in_range').show(10)

+----+----------+
|   T|T_in_range|
+----+----------+
|13.6|      true|
|13.3|      true|
|11.9|      true|
|11.0|      true|
|11.2|      true|
|11.2|      true|
|11.3|      true|
|10.7|      true|
|10.7|      true|
|10.3|      true|
+----+----------+
only showing top 10 rows


In [5]:
# Example 2 – only a lower bound: RH must be >= 0
air_obj.check_numeric_range('RH', lower=0)
air_obj.df.select('RH', 'RH_in_range').show(10)

+----+-----------+
|  RH|RH_in_range|
+----+-----------+
|48.9|       true|
|47.7|       true|
|54.0|       true|
|60.0|       true|
|59.6|       true|
|59.2|       true|
|56.8|       true|
|60.0|       true|
|59.7|       true|
|60.2|       true|
+----+-----------+
only showing top 10 rows


In [6]:
# Example 3 – only an upper bound: CO(GT) should not exceed 10
air_obj.check_numeric_range('CO(GT)', upper=10)
air_obj.df.select('CO(GT)', 'CO(GT)_in_range').show(10)

+------+---------------+
|CO(GT)|CO(GT)_in_range|
+------+---------------+
|   2.6|           true|
|   2.0|           true|
|   2.2|           true|
|   2.2|           true|
|   1.6|           true|
|   1.2|           true|
|   1.2|           true|
|   1.0|           true|
|   0.9|           true|
|   0.6|           true|
+------+---------------+
only showing top 10 rows


In [7]:
# Example 4 – neither bound supplied: should print a warning message
air_obj.check_numeric_range('AH')

check_numeric_range: please supply at least one of 'lower' or 'upper'.


In [8]:
# Example 5 – non-numeric column supplied: should print a message and leave df unchanged
air_obj.check_numeric_range('Date', lower=0, upper=100)

check_numeric_range: 'Date' is not a numeric column (type = 'string'). DataFrame was not modified.


#### `check_string_levels` – five examples

In [9]:
# Example 1 – check Date against a small known set of valid values
air_obj.check_string_levels('Date', levels=['3/10/2004', '3/11/2004', '3/12/2004'])
air_obj.df.select('Date', 'Date_valid_level').show(10)

+---------+----------------+
|     Date|Date_valid_level|
+---------+----------------+
|3/10/2004|            true|
|3/10/2004|            true|
|3/10/2004|            true|
|3/10/2004|            true|
|3/10/2004|            true|
|3/10/2004|            true|
|3/11/2004|            true|
|3/11/2004|            true|
|3/11/2004|            true|
|3/11/2004|            true|
+---------+----------------+
only showing top 10 rows


In [10]:
# Example 2 – custom result column name, checking Date against a wider set
air_obj.check_string_levels('Date',
                             levels=['3/10/2004', '3/11/2004', '3/12/2004',
                                     '3/13/2004', '3/14/2004'],
                             result_col='early_march')
air_obj.df.select('Date', 'early_march').show(10)

+---------+-----------+
|     Date|early_march|
+---------+-----------+
|3/10/2004|       true|
|3/10/2004|       true|
|3/10/2004|       true|
|3/10/2004|       true|
|3/10/2004|       true|
|3/10/2004|       true|
|3/11/2004|       true|
|3/11/2004|       true|
|3/11/2004|       true|
|3/11/2004|       true|
+---------+-----------+
only showing top 10 rows


In [11]:
# Example 3 – single valid level only
air_obj.check_string_levels('Date', levels=['3/10/2004'], result_col='first_day_flag')
air_obj.df.select('Date', 'first_day_flag').show(10)

+---------+--------------+
|     Date|first_day_flag|
+---------+--------------+
|3/10/2004|          true|
|3/10/2004|          true|
|3/10/2004|          true|
|3/10/2004|          true|
|3/10/2004|          true|
|3/10/2004|          true|
|3/11/2004|         false|
|3/11/2004|         false|
|3/11/2004|         false|
|3/11/2004|         false|
+---------+--------------+
only showing top 10 rows


In [12]:
# Example 4 – non-string column supplied: should print a message and leave df unchanged
air_obj.check_string_levels('T', levels=['warm', 'cold'])

check_string_levels: 'T' is not a string column (type = 'double'). DataFrame was not modified.


In [13]:
# Example 5 – chaining check_string_levels with check_missing
air_obj.check_string_levels('Date', levels=['3/10/2004']) \
       .check_missing('Date')
air_obj.df.select('Date', 'Date_valid_level', 'Date_is_missing').show(10)

+---------+----------------+---------------+
|     Date|Date_valid_level|Date_is_missing|
+---------+----------------+---------------+
|3/10/2004|            true|          false|
|3/10/2004|            true|          false|
|3/10/2004|            true|          false|
|3/10/2004|            true|          false|
|3/10/2004|            true|          false|
|3/10/2004|            true|          false|
|3/11/2004|           false|          false|
|3/11/2004|           false|          false|
|3/11/2004|           false|          false|
|3/11/2004|           false|          false|
+---------+----------------+---------------+
only showing top 10 rows


#### `check_missing` – five examples

In [14]:
# Example 1 – check for missing values in CO(GT)
air_obj.check_missing('CO(GT)')
air_obj.df.select('CO(GT)', 'CO(GT)_is_missing').show(10)

+------+-----------------+
|CO(GT)|CO(GT)_is_missing|
+------+-----------------+
|   2.6|            false|
|   2.0|            false|
|   2.2|            false|
|   2.2|            false|
|   1.6|            false|
|   1.2|            false|
|   1.2|            false|
|   1.0|            false|
|   0.9|            false|
|   0.6|            false|
+------+-----------------+
only showing top 10 rows


In [15]:
# Example 2 – count how many NULLs exist in the T column
air_obj.check_missing('T', result_col='T_null_flag')
air_obj.df.filter('T_null_flag = true').count()

0

In [16]:
# Example 3 – missing check on a string column
air_obj.check_missing('Date', result_col='date_missing')
air_obj.df.select('Date', 'date_missing').show(10)

+---------+------------+
|     Date|date_missing|
+---------+------------+
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
+---------+------------+
only showing top 10 rows


In [17]:
# Example 4 – chaining three validation calls together
air_obj.check_missing('RH') \
       .check_numeric_range('RH', lower=0, upper=100) \
       .check_missing('AH')
air_obj.df.select('RH', 'RH_is_missing', 'RH_in_range', 'AH', 'AH_is_missing').show(10)

+----+-------------+-----------+------+-------------+
|  RH|RH_is_missing|RH_in_range|    AH|AH_is_missing|
+----+-------------+-----------+------+-------------+
|48.9|        false|       true|0.7578|        false|
|47.7|        false|       true|0.7255|        false|
|54.0|        false|       true|0.7502|        false|
|60.0|        false|       true|0.7867|        false|
|59.6|        false|       true|0.7888|        false|
|59.2|        false|       true|0.7848|        false|
|56.8|        false|       true|0.7603|        false|
|60.0|        false|       true|0.7702|        false|
|59.7|        false|       true|0.7648|        false|
|60.2|        false|       true|0.7517|        false|
+----+-------------+-----------+------+-------------+
only showing top 10 rows


In [18]:
# Example 5 – custom result column name
air_obj.check_missing('NOx(GT)', result_col='NOx_null')
air_obj.df.select('NOx(GT)', 'NOx_null').show(10)

+-------+--------+
|NOx(GT)|NOx_null|
+-------+--------+
|    166|   false|
|    103|   false|
|    131|   false|
|    172|   false|
|    131|   false|
|     89|   false|
|     62|   false|
|     62|   false|
|     45|   false|
|   -200|   false|
+-------+--------+
only showing top 10 rows


#### `summarize_numeric` – five examples

In [19]:
# Example 1 – min/max of a single numeric column
air_obj.summarize_numeric('T')

,T_min,T_max
0,-200.0,44.6


In [20]:
# Example 2 – min/max of all numeric columns (no grouping)
air_obj.summarize_numeric()

26/03/26 09:31:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/26 09:31:29 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-emarti33@ncsu.edu/air.csv


,_c0_min,_c0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,...,PT08.S4(NO2)_min,PT08.S4(NO2)_max,PT08.S5(O3)_min,PT08.S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,0,9356,-200.0,11.9,-200,2040,-200,1189,-200.0,63.7,...,-200,2775,-200,2523,-200.0,44.6,-200.0,88.7,-200.0,2.231


In [21]:
# Example 3 – single column grouped by Date
air_obj.summarize_numeric('T', group_by='Date')

,Date,T_min,T_max
0,9/2/2004,21.0,38.7
1,12/26/2004,10.4,14.9
2,2/18/2005,5.4,11.8
3,10/10/2004,19.4,24.8
4,10/11/2004,17.9,23.2
...,...,...,...
386,1/23/2005,2.5,8.9
387,6/28/2004,22.8,40.4
388,8/16/2004,21.3,39.7
389,12/20/2004,6.8,9.8


In [22]:
# Example 4 – all numeric columns grouped by Date
air_obj.summarize_numeric(group_by='Date')

26/03/26 09:31:30 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date
 Schema: _c0, Date
Expected: _c0 but found: 
CSV file: file:///home/jupyter-emarti33@ncsu.edu/air.csv


,Date,_c0_min,_c0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,...,PT08.S4(NO2)_min,PT08.S4(NO2)_max,PT08.S5(O3)_min,PT08.S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,1/1/2005,7110,7133,-200.0,4.7,915,1472,-200,-200,3.0,...,897,1344,879,1735,2.6,12.8,32.3,63.2,0.4375,0.5961
1,1/10/2005,7326,7349,-200.0,3.5,958,1523,-200,-200,1.3,...,1122,1804,707,1634,12.3,14.4,63.6,73.3,0.9912,1.0567
2,1/11/2005,7350,7373,0.7,5.1,977,1526,-200,-200,2.1,...,1106,1855,830,1738,11.7,13.8,56.3,71.9,0.8754,1.0087
3,1/12/2005,7374,7397,-200.0,5.9,1002,1526,-200,-200,4.2,...,1139,1902,958,1770,8.7,15.8,48.8,79.4,0.8428,0.9467
4,1/13/2005,7398,7421,0.9,5.2,891,1523,-200,-200,3.2,...,1054,1811,798,1783,6.2,14.2,54.1,82.1,0.7751,0.9166
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386,9/5/2004,4278,4301,0.6,5.8,784,1399,-200,-200,1.9,...,1060,2052,342,1652,25.4,34.5,17.5,38.6,0.8162,1.2477
387,9/6/2004,4302,4325,-200.0,2.9,826,1133,-200,-200,1.0,...,1189,1669,385,1105,23.5,30.1,26.7,49.6,1.1217,1.4247
388,9/7/2004,4326,4349,-200.0,4.1,-200,1311,-200,-200,-200.0,...,-200,1957,-200,1486,-200.0,32.5,-200.0,48.3,-200.0000,1.3688
389,9/8/2004,4350,4373,-200.0,4.7,-200,1383,-200,-200,-200.0,...,-200,1582,-200,1678,-200.0,32.4,-200.0,33.0,-200.0000,1.0131


In [23]:
# Example 5 – non-numeric column supplied: should print a message and return None
result = air_obj.summarize_numeric('Date')
print(result)

summarize_numeric: 'Date' is not a numeric column. None returned.
None


#### `count_levels` – five examples

In [24]:
# Example 1 – count rows per unique Date
air_obj.count_levels('Date')

,Date,count
0,1/1/2005,24
1,1/10/2005,24
2,1/11/2005,24
3,1/12/2005,24
4,1/13/2005,24
...,...,...
386,9/5/2004,24
387,9/6/2004,24
388,9/7/2004,24
389,9/8/2004,24


In [25]:
# Example 2 – show first 10 results using head()
air_obj.count_levels('Date').head(10)

,Date,count
0,1/1/2005,24
1,1/10/2005,24
2,1/11/2005,24
3,1/12/2005,24
4,1/13/2005,24
5,1/14/2005,24
6,1/15/2005,24
7,1/16/2005,24
8,1/17/2005,24
9,1/18/2005,24


In [26]:
# Example 3 – timestamp column (Time) as second argument: should print a message and return None without modifying DataFrame
result = air_obj.count_levels('Date', col_name2='Time')
print(result)

count_levels: 'Time' is not a string column (type = 'timestamp'). None returned.
None


In [27]:
# Example 4 – numeric column as first argument: should print message, return None
result = air_obj.count_levels('T')
print(result)

count_levels: 'T' is not a string column (type = 'double'). None returned.
None


In [28]:
# Example 5 – numeric column as second argument: should print message, return None
result = air_obj.count_levels('Date', col_name2='T')
print(result)

count_levels: 'T' is not a string column (type = 'double'). None returned.
None


### 1.4 Create instance from a pandas DataFrame

Here we read `air.csv` with standard pandas, then convert it into a `SparkDataCheck` object using `from_pandas`.

In [29]:
import pandas as pd

air_pandas = pd.read_csv(AIR_PATH)
air_obj2   = SparkDataCheck.from_pandas(spark, air_pandas)
air_obj2.df.printSchema()

root
 |-- Unnamed: 0: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): long (nullable = true)
 |-- NMHC(GT): long (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): long (nullable = true)
 |-- NOx(GT): long (nullable = true)
 |-- PT08.S3(NOx): long (nullable = true)
 |-- NO2(GT): long (nullable = true)
 |-- PT08.S4(NO2): long (nullable = true)
 |-- PT08.S5(O3): long (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



In [30]:
# One example method call on the pandas-sourced object
air_obj2.check_numeric_range('T', lower=-5, upper=45)
air_obj2.df.select('T', 'T_in_range').show(10)

+----+----------+
|   T|T_in_range|
+----+----------+
|13.6|      true|
|13.3|      true|
|11.9|      true|
|11.0|      true|
|11.2|      true|
|11.2|      true|
|11.3|      true|
|10.7|      true|
|10.7|      true|
|10.3|      true|
+----+----------+
only showing top 10 rows


---
## Part II – NFL Quarterback Analysis

We analyze weekly NFL data from 2005–2023, focusing on quarterback (QB) performance during the regular season. The goal is to compute per-player, per-season aggregates and rank QBs by **completion percentage** and **TD-to-interception ratio**.

We perform this analysis twice — first using the **pandas-on-Spark** API, then using the **Spark SQL DataFrame** API — so we can directly compare their syntax and observe any differences in behavior (particularly around edge cases like division by zero).

### 2.1 pandas-on-Spark

The pandas-on-Spark API allows us to use familiar pandas-style syntax on top of a distributed Spark backend. We read the NFL data, filter it down to QB regular-season records from 2005–2023, aggregate by player and season, and compute two derived statistics.

In [31]:
spark.conf.set("spark.sql.ansi.enabled", "false")

import pyspark.pandas as ps

NFL_PATH = 'weekly_nfl_data.csv'

psdf = ps.read_csv(NFL_PATH)
print('Shape:', psdf.shape)

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


Shape: (128873, 53)


In [32]:
# First 5 rows
psdf.head(5)

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


In [33]:
# All column names
print(psdf.columns.tolist())

['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'recent_team', 'season', 'week', 'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'fantasy_points', 'fantasy_points_ppr']


In [34]:
# Filter: QBs, regular season, 2005-2023
qb_ps = psdf[
    (psdf['position']    == 'QB') &
    (psdf['season_type'] == 'REG') &
    (psdf['season']      >= 2005) &
    (psdf['season']      <= 2023)
]

# Subset to only the relevant columns
cols = ['player_display_name', 'season', 'week',
        'completions', 'attempts', 'passing_yards',
        'passing_tds', 'interceptions']
qb_ps = qb_ps[cols]
qb_ps.head(5)

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


In [35]:
# Aggregate by player + season: sum and mean of each statistical column
stat_cols = ['completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions']

agg_dict = {c: ['sum', 'mean'] for c in stat_cols}
qb_agg_ps = qb_ps.groupby(['player_display_name', 'season']).agg(agg_dict)

# Flatten multi-level column names (e.g. ('completions', 'sum') -> 'completions_sum')
qb_agg_ps.columns = ['_'.join(c).strip() for c in qb_agg_ps.columns]
qb_agg_ps = qb_agg_ps.reset_index()
qb_agg_ps.head(5)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000


In [36]:
# Create two new derived variables per player/season:
#   completion_percentage = sum of completions / sum of attempts
#   td_int_ratio          = sum of passing TDs / sum of interceptions
# NOTE: when interceptions_sum == 0, pandas-on-Spark returns NaN (not Infinity)
qb_agg_ps['completion_percentage'] = (
    qb_agg_ps['completions_sum'] / qb_agg_ps['attempts_sum']
)

qb_agg_ps['td_int_ratio'] = (
    qb_agg_ps['passing_tds_sum'] / qb_agg_ps['interceptions_sum']
)

qb_agg_ps.head(5)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,0.610209,1.545455
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,0.607456,2.571429
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000,0.666667,0.500000
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,0.516854,0.923077
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,0.609756,NaN


In [37]:
# Keep only player/season combinations with at least 50 attempts
qb_filtered_ps = qb_agg_ps[qb_agg_ps['attempts_sum'] >= 50]

# Top 40 by completion percentage (descending)
print('=== Top 40 by Completion Percentage (pandas-on-Spark) ===')
(
    qb_filtered_ps
    .sort_values('completion_percentage', ascending=False)
    [['player_display_name', 'season', 'completions_sum', 'attempts_sum', 'completion_percentage']]
    .head(40)
)

=== Top 40 by Completion Percentage (pandas-on-Spark) ===


,player_display_name,season,completions_sum,attempts_sum,completion_percentage
1409,C.J. Beathard,2023,40,53,0.754717
1248,Colt McCoy,2021,74,99,0.747475
900,Matt Schaub,2019,50,67,0.746269
953,Drew Brees,2018,364,489,0.744376
1057,Drew Brees,2019,281,378,0.743386
1338,Mason Rudolph,2023,55,74,0.743243
1133,Taysom Hill,2020,88,121,0.727273
1007,Nick Foles,2018,141,195,0.723077
917,Drew Brees,2017,386,536,0.720149
851,Sam Bradford,2016,395,552,0.715580


In [38]:
# Top 40 by TD/INT ratio (descending)
# NaN values (from 0 interceptions) are sorted to the bottom with na_position='last'
print('=== Top 40 by TD-INT Ratio (pandas-on-Spark) ===')
(
    qb_filtered_ps
    .sort_values('td_int_ratio', ascending=False, na_position='last')
    [['player_display_name', 'season', 'passing_tds_sum', 'interceptions_sum', 'td_int_ratio']]
    .head(40)
)

=== Top 40 by TD-INT Ratio (pandas-on-Spark) ===


,player_display_name,season,passing_tds_sum,interceptions_sum,td_int_ratio
6,Charlie Batch,2006,5,0.0,inf
26,Matt Schaub,2005,4,0.0,inf
73,Todd Collins,2007,5,0.0,inf
455,Troy Smith,2007,2,0.0,inf
520,Jake Locker,2011,4,0.0,inf
775,Brian Hoyer,2016,6,0.0,inf
778,Nick Foles,2016,3,0.0,inf
812,Derek Anderson,2014,5,0.0,inf
940,Jimmy Garoppolo,2016,4,0.0,inf
984,Matt Moore,2019,4,0.0,inf


**Interpretation (pandas-on-Spark results):**  
The top completion-percentage seasons are dominated by modern QBs (2010s–2020s), reflecting rule changes that favor passing. The TD/INT ratio list highlights elite, low-turnover seasons. Note that when `interceptions_sum == 0`, pandas-on-Spark produces `NaN` and those rows are pushed to the bottom of the TD/INT ranking.

> **Note:** Update this section with specific player names and seasons from your actual output once you have run the notebook.

### 2.2 Spark SQL DataFrame API

We now repeat the exact same analysis using the Spark SQL DataFrame API. The logic and steps are identical to the pandas-on-Spark section above, but the syntax is different— we use `filter()`, `select()`, `groupBy().agg()`, and `withColumn()` instead of pandas-style indexing. This also lets us observe one important behavioral difference: how each API handles division by zero.

In [39]:
from pyspark.sql import functions as F

# Read the NFL data with Spark SQL
nfl_sdf = spark.read.load(
    NFL_PATH,
    format='csv',
    sep=',',
    inferSchema='true',
    header='true'
)

nfl_sdf.printSchema()

root
 |-- player_id: string (nullable = true)
 |-- player_name: string (nullable = true)
 |-- player_display_name: string (nullable = true)
 |-- position: string (nullable = true)
 |-- position_group: string (nullable = true)
 |-- headshot_url: string (nullable = true)
 |-- recent_team: string (nullable = true)
 |-- season: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- season_type: string (nullable = true)
 |-- opponent_team: string (nullable = true)
 |-- completions: integer (nullable = true)
 |-- attempts: integer (nullable = true)
 |-- passing_yards: double (nullable = true)
 |-- passing_tds: integer (nullable = true)
 |-- interceptions: double (nullable = true)
 |-- sacks: double (nullable = true)
 |-- sack_yards: double (nullable = true)
 |-- sack_fumbles: integer (nullable = true)
 |-- sack_fumbles_lost: integer (nullable = true)
 |-- passing_air_yards: double (nullable = true)
 |-- passing_yards_after_catch: double (nullable = true)
 |-- passing_first_downs

In [40]:
# First 5 rows
nfl_sdf.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

In [41]:
# All column names
print(nfl_sdf.columns)

['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'recent_team', 'season', 'week', 'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'fantasy_points', 'fantasy_points_ppr']


In [42]:
# Filter: QBs, regular season, 2005-2023
qb_sdf = nfl_sdf.filter(
    (F.col('position')    == 'QB') &
    (F.col('season_type') == 'REG') &
    (F.col('season')      >= 2005) &
    (F.col('season')      <= 2023)
)

# Subset to only the relevant columns
qb_sdf = qb_sdf.select(
    'player_display_name', 'season', 'week',
    'completions', 'attempts', 'passing_yards',
    'passing_tds', 'interceptions'
)
qb_sdf.show(5)

+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|player_display_name|season|week|completions|attempts|passing_yards|passing_tds|interceptions|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|         Tony Banks|  2005|  17|         14|      25|        173.0|          1|          2.0|
|      Charlie Batch|  2005|   9|          9|      16|         65.0|          0|          1.0|
|      Charlie Batch|  2005|  10|         13|      19|        150.0|          0|          0.0|
|      Charlie Batch|  2005|  16|          1|       1|         31.0|          1|          0.0|
|         Jeff Blake|  2005|   2|          1|       1|         11.0|          0|          0.0|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
only showing top 5 rows


In [43]:
# Aggregate by player + season: sum and mean of each statistical column
qb_agg_sdf = qb_sdf.groupBy('player_display_name', 'season').agg(
    F.sum('completions').alias('completions_sum'),
    F.mean('completions').alias('completions_mean'),
    F.sum('attempts').alias('attempts_sum'),
    F.mean('attempts').alias('attempts_mean'),
    F.sum('passing_yards').alias('passing_yards_sum'),
    F.mean('passing_yards').alias('passing_yards_mean'),
    F.sum('passing_tds').alias('passing_tds_sum'),
    F.mean('passing_tds').alias('passing_tds_mean'),
    F.sum('interceptions').alias('interceptions_sum'),
    F.mean('interceptions').alias('interceptions_mean')
)

qb_agg_sdf.show(5)

+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|  passing_tds_mean|interceptions_sum|interceptions_mean|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+
|      Jake Delhomme|  2006|            263| 20.23076923076923|         431| 33.15384615384615|           2805.0|215.76923076923077|             17|1.3076923076923077|             11.0|0.8461538461538461|
|       Jake Plummer|  2005|            277|           17.3125|         456|              28.5|           3366.0|           210.375|             18|             1.125|             

In [44]:
# Create two new derived variables per player/season:
#   completion_percentage = sum of completions / sum of attempts
#   td_int_ratio          = sum of passing TDs / sum of interceptions
# NOTE: when interceptions_sum == 0, Spark SQL returns Infinity (not NaN like pandas-on-Spark)
qb_agg_sdf = qb_agg_sdf.withColumn(
    'completion_percentage',
    F.col('completions_sum') / F.col('attempts_sum')
).withColumn(
    'td_int_ratio',
    F.col('passing_tds_sum') / F.col('interceptions_sum')
)

qb_agg_sdf.show(5)

+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+---------------------+------------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|  passing_tds_mean|interceptions_sum|interceptions_mean|completion_percentage|      td_int_ratio|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|            263| 20.23076923076923|         431| 33.15384615384615|           2805.0|215.76923076923077|             17|1.3076923076923077|             11.0|0.8461538461538461|   0.6102088167053364|1.5454545454545454|
|       Jake Plu

In [45]:
# Keep only player/season combinations with at least 50 attempts
qb_filtered_sdf = qb_agg_sdf.filter(F.col('attempts_sum') >= 50)

# Top 40 by completion percentage (descending)
print('=== Top 40 by Completion Percentage (Spark SQL) ===')
qb_filtered_sdf \
    .orderBy(F.col('completion_percentage').desc()) \
    .select('player_display_name', 'season', 'completions_sum',
            'attempts_sum', 'completion_percentage') \
    .show(40, truncate=False)

=== Top 40 by Completion Percentage (Spark SQL) ===
+-------------------+------+---------------+------------+---------------------+
|player_display_name|season|completions_sum|attempts_sum|completion_percentage|
+-------------------+------+---------------+------------+---------------------+
|C.J. Beathard      |2023  |40             |53          |0.7547169811320755   |
|Colt McCoy         |2021  |74             |99          |0.7474747474747475   |
|Matt Schaub        |2019  |50             |67          |0.746268656716418    |
|Drew Brees         |2018  |364            |489         |0.7443762781186094   |
|Drew Brees         |2019  |281            |378         |0.7433862433862434   |
|Mason Rudolph      |2023  |55             |74          |0.7432432432432432   |
|Taysom Hill        |2020  |88             |121         |0.7272727272727273   |
|Nick Foles         |2018  |141            |195         |0.7230769230769231   |
|Drew Brees         |2017  |386            |536         |0.720149253

In [46]:
# Top 40 by TD/INT ratio (descending)
# IMPORTANT: Spark SQL returns Infinity for 0-interception seasons, which sort to the TOP.
# This differs from pandas-on-Spark where those rows produce NaN and sort to the BOTTOM.
print('=== Top 40 by TD-INT Ratio (Spark SQL) ===')
print('NOTE: Rows with 0 interceptions produce Infinity here, vs. NaN in pandas-on-Spark.')
qb_filtered_sdf \
    .orderBy(F.col('td_int_ratio').desc()) \
    .select('player_display_name', 'season', 'passing_tds_sum',
            'interceptions_sum', 'td_int_ratio') \
    .show(40, truncate=False)

=== Top 40 by TD-INT Ratio (Spark SQL) ===
NOTE: Rows with 0 interceptions produce Infinity here, vs. NaN in pandas-on-Spark.
+-------------------+------+---------------+-----------------+-----------------+
|player_display_name|season|passing_tds_sum|interceptions_sum|td_int_ratio     |
+-------------------+------+---------------+-----------------+-----------------+
|Tom Brady          |2016  |28             |2.0              |14.0             |
|Nick Foles         |2013  |27             |2.0              |13.5             |
|Josh McCown        |2013  |13             |1.0              |13.0             |
|Aaron Rodgers      |2018  |25             |2.0              |12.5             |
|Damon Huard        |2006  |11             |1.0              |11.0             |
|Aaron Rodgers      |2020  |48             |5.0              |9.6              |
|Aaron Rodgers      |2021  |37             |4.0              |9.25             |
|Tom Brady          |2010  |36             |4.0              |9.

**Key difference – division by zero:**  
When a QB recorded zero interceptions in a season, dividing TDs by 0 gives:  
- **pandas-on-Spark** → `NaN` (rows sort to the bottom when sorting descending)  
- **Spark SQL DataFrame** → `Infinity` (rows sort to the top when sorting descending)  

This means the TD/INT ratio top-40 lists look different between the two APIs at the top of the ranking— not because the data differs, but because of how each handles division by zero. Both APIs agree on all rows where interceptions > 0.

**Overall findings:**  
The completion-percentage rankings are consistent between both APIs, confirming correctness. The TD/INT ratio list rewards seasons with many touchdowns and very few turnovers — a hallmark of elite, efficient QB play.

In [47]:
spark.stop()